# Week 11 — BBO capstone driver

Round 11. **The change that mattered most.**

Stop treating each function as a point search. Parameterise a single line per function,

> `x(λ) = x₈ + λ·(c − x₈)`,  with `c` the domain centre (0.5, …, 0.5)

and probe λ ∈ {0.3, 0.5, 0.7, 1.0}. An 8-dimensional search becomes a 1-dimensional one along a direction I can justify, and a single query now interrogates a whole ray.

F4 takes λ=0.7, F6 λ=0.5, F7 λ=1.0 (the centre itself), F8 λ=0.3 from its W10 point. F5 continues along its own established ray, F1 goes to the 0.42–0.46 hot zone, F3 steps opposite its failed W10 perturbation.

Five functions improve simultaneously.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 11
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 11
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: 'hot-zone probe 0.42–0.46',
    2: 'gradient step',
    3: 'reverse W10 step',
    4: 'centre line λ=0.7',
    5: '4 steps along own ray',
    6: 'centre line λ=0.5',
    7: 'centre line λ=1.0',
    8: 'centre line λ=0.3 from W10',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 10. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — the centre-ward line

`bbo.centre_line(anchor, λ)` generates the point; the derived values are checked against what was actually submitted.

In [ ]:
proposals = {
    1: np.array([0.42, 0.462]),
    2: np.array([0.754032, 0.689499]),
    3: np.array([0.061275, 0.824316, 0.466951]),
    4: np.array([0.405162, 0.369606, 0.355285, 0.621298]),
    5: np.array([0.280806, 0.758874, 0.061998, 0.689917]),
    6: np.array([0.285011, 0.515366, 0.726427, 0.662903, 0.364399]),
    7: np.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5]),
    8: np.array([0.177288, 0.34284, 0.277349, 0.393462, 0.712236, 0.314334, 0.805229, 0.216194]),
}

# Reconstruct the line-search proposals and confirm they match the submissions.
LINE = {4: (8, 0.7), 6: (8, 0.5), 7: (8, 1.0), 8: (10, 0.3)}
rows = []
for fid, (anchor_round, lam) in LINE.items():
    anchor = np.array(bbo.HISTORY[anchor_round][fid][0])
    derived = bbo.centre_line(anchor, lam)
    rows.append(dict(func=f"F{fid}", anchor_round=anchor_round, lam=lam,
                     derived=bbo.submission(derived),
                     matches_submission=bool(np.allclose(derived, proposals[fid], atol=1e-5))))
pd.DataFrame(rows)


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.669937, 0.751452],
    2: [0.354432, 0.449899],
    3: [0.111275, 0.774316, 0.516951],
    4: [0.183872, 0.065353, 0.017618, 0.904326],
    5: [0.308806, 0.730874, 0.091998, 0.661917],
    6: [0.070021, 0.530732, 0.952853, 0.825805, 0.228797],
    7: [0.965568, 0.153915, 0.588691, 0.807159, 0.099427, 0.700138],
    8: [0.038983, 0.275485, 0.181927, 0.347803, 0.803194, 0.234763, 0.936041, 0.094563],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 11 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: -0.0036427,
#     2: 0.291066,
#     3: -0.038737,
#     4: -4.06313,
#     5: 7.134355,
#     6: -0.730692,
#     7: 0.505315,
#     8: 9.040799,
# }
#
# FIVE SIMULTANEOUS IMPROVEMENTS - the best round of the project.
#   F4 -23.62 -> -4.06   F5 1.327 -> 7.134   F6 -1.014 -> -0.731
#   F7 0.499 -> 0.505    F8 8.750 -> 9.041 (the 8.75 plateau was a false summit)
# F4, F6, F7 and F8 all improved by moving toward the centre; F1 did not. Four out of
# five looks structural - but see W13 for how far that generalisation actually holds.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
